# 🏋️ Local Machine Model Training & Entity Resolution Pipeline

This notebook runs the full ML training and inference pipeline locally using **pre-cleaned datasets**.
- Optimized for **16GB RAM** and **NVIDIA RTX 2050 GPU (4GB VRAM)**.
- Input cleaned files: `dataset/cleaned/` (`train_source1_clean.tsv`, etc.)
- Ground truth file: `dataset/train/train_ground_truth.tsv`
- Output directory: `output/` (`matching_results.tsv` and `candidate_pairs.tsv`)


---
## 📦 Cell 1: Install Project Dependencies

In [ ]:
import os, sys

# Auto-detect Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Google Colab environment detected.')
    if not os.path.exists('Error_404_Not_Found_submission'):
        if os.path.exists('ml_challenge/Error_404_Not_Found_submission'):
            %cd ml_challenge
        else:
            print('Cloning repository into Colab...')
            !git clone https://github.com/BharathPESU/ml_challenge.git
            %cd ml_challenge
            !git checkout Bharath.2004

# Locate requirements file
req_paths = [
    os.path.abspath('Error_404_Not_Found_submission/code/business_entity_resolution/requirements.txt'),
    os.path.abspath('code/business_entity_resolution/requirements.txt'),
    os.path.abspath('requirements.txt'),
]
req_path = next((p for p in req_paths if os.path.exists(p)), None)

if req_path:
    print(f'Installing dependencies from: {req_path}...')
    %pip install -r "{req_path}" --quiet
else:
    print('Installing core dependencies (pandas, numpy, rapidfuzz, xgboost, scikit-learn, pyyaml)...')
    %pip install pandas numpy rapidfuzz xgboost scikit-learn pyyaml --quiet

print('All required packages installed successfully!')

---
## ⚙️ Cell 2: Imports, Path Setup & Hardware Configuration

In [ ]:
import os, sys, logging, glob
import pandas as pd
from IPython.display import display

# Add src directory to Python search path dynamically
candidate_src_paths = [
    os.path.abspath('Error_404_Not_Found_submission/code/business_entity_resolution/src'),
    os.path.abspath('code/business_entity_resolution/src'),
    os.path.abspath('src'),
]
if not any(os.path.exists(p) for p in candidate_src_paths):
    matches = glob.glob('**/business_entity_resolution/src', recursive=True)
    if matches:
        candidate_src_paths.insert(0, os.path.abspath(matches[0]))

for p in candidate_src_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from config import Config
from pipeline import PipelineState

# Auto-detect raw dataset root for train_ground_truth.tsv
dataset_candidates = [
    os.path.abspath('Error_404_Not_Found_submission/dataset'),
    os.path.abspath('dataset'),
    os.path.abspath('../Error_404_Not_Found_submission/dataset'),
    os.path.abspath('student_resource/dataset'),
]
RAW_DATASET_ROOT = None
for cand in dataset_candidates:
    if os.path.exists(os.path.join(cand, 'train', 'train_ground_truth.tsv')):
        RAW_DATASET_ROOT = cand
        break
if RAW_DATASET_ROOT is None:
    found = glob.glob('**/train_ground_truth.tsv', recursive=True)
    if found:
        RAW_DATASET_ROOT = os.path.abspath(os.path.dirname(os.path.dirname(found[0])))
    else:
        RAW_DATASET_ROOT = os.path.abspath('Error_404_Not_Found_submission/dataset')

CLEANED_DIR = os.path.abspath('dataset/cleaned')
TRAIN_DIR = os.path.join(RAW_DATASET_ROOT, 'train')
OUTPUT_DIR = os.path.abspath('output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize configuration
cfg = Config()
cfg.data.train_source1_path = os.path.join(CLEANED_DIR, 'train_source1_clean.tsv')
cfg.data.train_source2_path = os.path.join(CLEANED_DIR, 'train_source2_clean.tsv')
cfg.data.train_source3_path = os.path.join(CLEANED_DIR, 'train_source3_clean.tsv')
cfg.data.train_ground_truth_path = os.path.join(TRAIN_DIR, 'train_ground_truth.tsv')

cfg.data.test_source1_path = os.path.join(CLEANED_DIR, 'test_source1_clean.tsv')
cfg.data.test_source2_path = os.path.join(CLEANED_DIR, 'test_source2_clean.tsv')
cfg.data.test_source3_path = os.path.join(CLEANED_DIR, 'test_source3_clean.tsv')
cfg.data.output_dir = OUTPUT_DIR

# Optimized for local hardware: 16GB RAM & RTX 2050 GPU (4GB VRAM)
cfg.xgboost.tree_method = 'hist'
cfg.xgboost.device = 'cuda:0'
cfg.runtime.debug_sample_size = None  # Full dataset run

# Configure logging & create shared pipeline state
cfg.configure_logging()
state = PipelineState()

print('Configuration initialized for local laptop execution')
print(f'   Cleaned datasets    : {CLEANED_DIR}')
print(f'   Ground truth path   : {cfg.data.train_ground_truth_path} (exists: {os.path.exists(cfg.data.train_ground_truth_path)})')
print(f'   Output directory    : {OUTPUT_DIR}')
print(f'   XGBoost device      : {cfg.xgboost.device} ({cfg.xgboost.tree_method})')

---
## 📥 Cell 3: Load Pre-Cleaned Training Datasets & Ground Truth

In [ ]:
from io_utils import read_tsv, validate_source_file, validate_ground_truth_file
from ground_truth import parse_ground_truth

print("⏳ Loading pre-cleaned training datasets...")
state.train_s1_clean = read_tsv(cfg.data.train_source1_path)
state.train_s2_clean = read_tsv(cfg.data.train_source2_path)
state.train_s3_clean = read_tsv(cfg.data.train_source3_path)

# Reference clean DataFrames as primary sources
state.train_s1 = state.train_s1_clean
state.train_s2 = state.train_s2_clean
state.train_s3 = state.train_s3_clean

validate_source_file(state.train_s1_clean, "train_source1_clean")
validate_source_file(state.train_s2_clean, "train_source2_clean")
validate_source_file(state.train_s3_clean, "train_source3_clean")

gt_df = read_tsv(cfg.data.train_ground_truth_path)
validate_ground_truth_file(gt_df)
state.gt = parse_ground_truth(gt_df)

assert state.gt is not None
print(f"✅ Loaded pre-cleaned data:")
print(f"   S1 Clean: {len(state.train_s1_clean):,} rows")
print(f"   S2 Clean: {len(state.train_s2_clean):,} rows")
print(f"   S3 Clean: {len(state.train_s3_clean):,} rows")
print(f"   GT      : {len(state.gt):,} entities")

---
## ✂️ Cell 4 — Stage 3: Train/Validation Split

In [ ]:
from pipeline import split_training_validation

print("⏳ Stage 3: Splitting into train and validation sets (80/20)... ")
split_training_validation(cfg, state)
assert state.train_s1_ids is not None and state.val_s1_ids is not None
print(f"✅ Stage 3 Complete — Train S1 IDs: {len(state.train_s1_ids):,} | Val S1 IDs: {len(state.val_s1_ids):,}")

---
## 🔍 Cell 5 — Stage 4: Generate Candidate Pairs (Multi-Pass Blocking)

In [ ]:
from pipeline import generate_candidates

print("⏳ Stage 4: Running 8-pass TF-IDF & token blocking...")
generate_candidates(cfg, state)
assert state.train_candidates is not None and state.val_candidates is not None
print(f"✅ Stage 4 Complete — Train candidates: {len(state.train_candidates):,} | Val candidates: {len(state.val_candidates):,}")

---
## 📊 Cell 6 — Stage 5: Evaluate Candidate Recall

In [ ]:
from pipeline import evaluate_candidate_recall

print("⏳ Stage 5: Evaluating candidate blocking recall...")
evaluate_candidate_recall(cfg, state)
print("✅ Stage 5 Complete — See logs above for train and val recall percentages.")

---
## 🏗️ Cell 7 — Stage 6: Build Training Pairs & Hard Negatives

In [ ]:
from pipeline import build_training_pairs

print("⏳ Stage 6: Building labeled training and validation pair sets...")
build_training_pairs(cfg, state)
assert state.train_pairs is not None and state.val_pairs is not None
print(f"✅ Stage 6 Complete — Train pairs: {len(state.train_pairs):,} | Val pairs: {len(state.val_pairs):,}")

---
## 🧮 Cell 8 — Stage 7: Extract Similarity Features

In [ ]:
from pipeline import build_features

print("⏳ Stage 7: Computing 30+ pairwise feature similarities...")
build_features(cfg, state)
assert state.feature_columns is not None
print(f"✅ Stage 7 Complete — Feature columns ({len(state.feature_columns)}): {state.feature_columns[:5]}...")

---
## 🤖 Cell 9 — Stage 8: Train XGBoost Classifier

In [ ]:
from pipeline import train_xgboost

print("⏳ Stage 8: Training XGBoost model (GPU accelerated)... ")
try:
    train_xgboost(cfg, state)
except Exception as err:
    print(f"⚠️ GPU execution encountered error: {err}. Retrying with CPU execution...")
    cfg.xgboost.tree_method = "hist"
    cfg.xgboost.device = "cpu"
    train_xgboost(cfg, state)

print(f"✅ Stage 8 Complete — Model successfully trained & saved to: {state.model_path}")

---
## 📈 Cell 10 — Stage 9: Score Validation Set

In [ ]:
from pipeline import score_validation

print("⏳ Stage 9: Scoring validation candidates...")
score_validation(cfg, state)
assert state.val_probabilities is not None
print(f"✅ Stage 9 Complete — Generated probabilities for {len(state.val_probabilities):,} validation candidates")

---
## 🎯 Cell 11 — Stage 10: Sweep Thresholds for Optimal Macro F0.5

In [ ]:
from pipeline import optimize_threshold

print("⏳ Stage 10: Sweeping threshold from 0.05 to 0.95 for maximum Macro F0.5...")
optimize_threshold(cfg, state)
assert state.best_threshold is not None
print(f"✅ Stage 10 Complete — Optimal threshold: {state.best_threshold:.4f}")

---
## 🔎 Cell 12 — Stage 11: Error Analysis

In [ ]:
from pipeline import perform_error_analysis

print("⏳ Stage 11: Performing error analysis (False Positives & False Negatives)...")
perform_error_analysis(cfg, state)
print("✅ Stage 11 Complete — Error analysis breakdown logged above.")

---
## 🧪 Cell 13 — Stage 12: Generate Test Candidates (Pre-Cleaned Test Data)

In [ ]:
from inference import generate_test_candidates as _gen_test_cands
from io_utils import read_tsv

print("⏳ Stage 12: Loading pre-cleaned test data and generating test candidate pairs...")
state.test_s1_clean = read_tsv(cfg.data.test_source1_path)
state.test_s2_clean = read_tsv(cfg.data.test_source2_path)
state.test_s3_clean = read_tsv(cfg.data.test_source3_path)

bc = cfg.blocking
blocking_config = {
    "name_top_k": bc.name_top_k,
    "address_top_k": bc.address_top_k,
    "min_name_similarity": bc.min_name_similarity,
    "min_address_similarity": bc.min_address_similarity,
    "tfidf_analyzer": bc.tfidf_analyzer,
    "tfidf_ngram_range": list(bc.tfidf_ngram_range),
    "tfidf_min_df": bc.tfidf_min_df,
    "tfidf_sublinear_tf": bc.tfidf_sublinear_tf,
    "use_token_blocking": True,
}

state.test_candidates = _gen_test_cands(
    state.test_s1_clean, state.test_s2_clean, state.test_s3_clean,
    blocking_config=blocking_config,
)

assert state.test_s1_clean is not None and state.test_candidates is not None
print(f"✅ Stage 12 Complete — Test S1: {len(state.test_s1_clean):,} | Test Candidates: {len(state.test_candidates):,}")

---
## 🧮 Cell 14 — Stage 13: Build Test Features

In [ ]:
from pipeline import build_test_features

print("⏳ Stage 13: Computing test feature similarities...")
build_test_features(cfg, state)
assert state.test_features is not None
print(f"✅ Stage 13 Complete — Test Feature Matrix: {state.test_features.shape}")

---
## 🎯 Cell 15 — Stage 14: Run Test Inference

In [ ]:
from pipeline import run_test_inference

print("⏳ Stage 14: Scoring test candidates with trained XGBoost model...")
run_test_inference(cfg, state)
assert state.test_predictions is not None
n_matched = sum(1 for v in state.test_predictions.values() if v)
print(f"✅ Stage 14 Complete — Predictions generated for {len(state.test_predictions):,} S1 entities ({n_matched:,} with matches)")

---
## 📝 Cell 16 — Stage 15: Create Submission TSV Files

In [ ]:
from pipeline import create_submission_files

print("⏳ Stage 15: Writing matching_results.tsv and candidate_pairs.tsv...")
create_submission_files(cfg, state)
print(f"\n🎉 Stage 15 Complete — Submission files written into {cfg.data.output_dir}!")
print(f"   📄 {cfg.data.output_dir}/matching_results.tsv")
print(f"   📄 {cfg.data.output_dir}/candidate_pairs.tsv")

---
## ✅ Cell 17: Inspect & Validate Output TSV Files

In [ ]:
import pandas as pd
from IPython.display import display

matching_tsv = os.path.join(cfg.data.output_dir, "matching_results.tsv")
candidate_tsv = os.path.join(cfg.data.output_dir, "candidate_pairs.tsv")

if os.path.exists(matching_tsv) and os.path.exists(candidate_tsv):
    m_df = pd.read_csv(matching_tsv, sep="\t", dtype=str, keep_default_na=False)
    c_df = pd.read_csv(candidate_tsv, sep="\t", dtype=str, keep_default_na=False)

    print("📊 matching_results.tsv Summary:")
    print(f"   Total S1 Entities   : {len(m_df):,}")
    print(f"   Entities Matched     : {(m_df['matched_entity_ids'] != '').sum():,}")
    print(f"   Singletons           : {(m_df['matched_entity_ids'] == '').sum():,}")
    print("-" * 60)
    print("📊 candidate_pairs.tsv Summary:")
    print(f"   Total Candidate Pairs: {len(c_df):,}")
    print("-" * 60)
    print("\n🔍 First 10 rows of matching_results.tsv:")
    display(m_df.head(10))
else:
    print("⚠️ Output submission files not found. Ensure Stage 15 completed.")